In [ ]:
"""
=============================================================================
  TEMPORAL EPIDEMIOLOGICAL ANALYSIS — Phytophthora Root Rot (PRR) in Avocado
=============================================================================
  Study area : Three commercial avocado lots, Antioquia, Colombia
  Data source : Field survey records — real plant-level severity evaluations
  Scale       : 0 = healthy · 1–4 = progressive infection stages · 5 = dead

  Lots (read from a single unified workbook)
  ------------------------------------------
    Sheet "Donmatias-L1"  →  903 plants   (L1)
    Sheet "El Retiro-L2"  →  1 338 plants (L2)
    Sheet "La Ceja-L3"    →  1 666 plants (L3)

  Workbook structure per sheet
  ----------------------------
    Row 1    : column header  (Lot_ID, Lot_Label, GPS_Height, Vert_Prec,
                                Horz_Prec, Northing, Easting, Point_ID,
                                ORIG_FID, D0, D60, D120, … D1200)
    Row 2+   : plant records

  Pipeline
  --------
    1  Load all lots from the unified workbook
    2  Compute cumulative epidemiological curves per time point:
         · Incidence  (% plants with severity ≥ 1)
         · AUDPC      (trapezoidal rule, averaged across plants)
         · Mortality  (% plants with severity = 5)
         · Rate of change per 60-day interval for each variable
    3  Fit six classical epidemiological models to incidence and AUDPC:
         Monomolecular · Exponential · Logistic · Gompertz · Weibull · Log-logistic
    4  Generate quality figures (500 dpi PNG + PDF vector)
    5  Export model-selection metrics to Word table and CSV

  Outputs  (written to OUTPUT_DIR)
  ---------------------------------
    Fig1_Cumulative.png / .pdf   3×3 cumulative curves + rate of change
    Fig2_ModelFits.png  / .pdf   2×3 model fits (A = Incidence, B = AUDPC)
    Table_ModelFits.docx         R², RMSE, AIC per lot × model
    model_fit_metrics.csv        raw metrics for downstream analysis

  Dependencies
  ------------
    pip install pandas numpy scipy matplotlib openpyxl python-docx

  Usage
  -----
    1. Place PRR_Unified_data_ThreeLots.xlsx in the same folder as this script.
    2. python temporal_analysis.ipynb
=============================================================================
"""

import os
import warnings
import numpy  as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import matplotlib.ticker    as mticker
from matplotlib.lines import Line2D
warnings.filterwarnings("ignore")


# =============================================================================
# 0.  CONFIGURATION  ← only edit this block
# =============================================================================

# Path to the unified workbook (one sheet per lot)
UNIFIED_XLSX  = "PRR_Unified_data_ThreeLots.xlsx"

# Sheet names — must match the workbook exactly
SHEETS = {
    "Donmatias-L1" : "Donmatias-L1",
    "El Retiro-L2" : "El Retiro-L2",
    "La Ceja-L3"   : "La Ceja-L3",
}

METADATA_ROWS = 6

OUTPUT_DIR          = "outputs"
TIME_POINTS         = list(range(0, 1201, 60))   # D0 … D1200
INCIDENCE_THRESHOLD = 1    # severity ≥ 1  → infected
MORTALITY_THRESHOLD = 5    # severity = 5  → dead
DPI                 = 500

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# 1.  DATA LOADING
# =============================================================================

def load_lot(xlsx_path: str, sheet_name: str, label: str) -> pd.DataFrame:
    """
    Load one lot from the unified workbook.

    Parameters
    ----------
    xlsx_path  : path to PRR_Unified_data_ThreeLots.xlsx
    sheet_name : exact sheet name for this lot
    label      : display label used in logs and figures

    Returns
    -------
    DataFrame with columns:
        Lot_ID · Lot_Label · GPS_Height · Vert_Prec · Horz_Prec ·
        Northing · Easting · Point_ID · ORIG_FID · D0 · D60 · … · D1200
    """
    df = pd.read_excel(xlsx_path, sheet_name=sheet_name, skiprows=METADATA_ROWS)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(how="all").reset_index(drop=True)

    n_time = sum(1 for c in df.columns
                 if c.startswith("D") and c[1:].isdigit())
    print(f"  [{label}]  {len(df):>5} plants  |  {n_time} time columns  |  "
          f"Northing {df['Northing'].min():.0f}–{df['Northing'].max():.0f} m  |  "
          f"Easting {df['Easting'].min():.0f}–{df['Easting'].max():.0f} m")
    return df


def load_all_lots() -> dict:
    """
    Load every lot from UNIFIED_XLSX and return an ordered dict {label: df}.
    Raises RuntimeError if the workbook cannot be found.
    """
    if not os.path.exists(UNIFIED_XLSX):
        raise RuntimeError(
            f"Workbook not found: '{UNIFIED_XLSX}'\n"
            f"Place the file in: {os.path.abspath('.')}")

    print(f"\nReading workbook: {UNIFIED_XLSX}")
    dfs = {}
    for label, sheet in SHEETS.items():
        try:
            dfs[label] = load_lot(UNIFIED_XLSX, sheet, label)
        except Exception as exc:
            print(f"  [{label}]  ERROR reading sheet '{sheet}': {exc}")
    return dfs


# =============================================================================
# 2.  EPIDEMIOLOGICAL CURVE COMPUTATION
# =============================================================================

def compute_curves(df: pd.DataFrame, label: str) -> dict:
    """
    Compute cumulative incidence, AUDPC, mortality, and 60-day rates.

    Severity scale (0–5)
    --------------------
      0    healthy
      1–4  progressive root-rot stages (feeder-root loss → crown collapse)
      5    plant dead

    Returns
    -------
    dict with keys:
        label · months · inc · audpc · mort ·
        rate_inc · rate_audpc · rate_mort
    """
    dcols  = [f"D{t}" for t in TIME_POINTS if f"D{t}" in df.columns]
    n      = len(df)
    sev    = df[dcols].values.astype(float)
    months = [t / 30 for t in TIME_POINTS]

    # Cumulative incidence and mortality (%)
    inc  = (sev >= INCIDENCE_THRESHOLD).mean(axis=0) * 100
    mort = (sev >= MORTALITY_THRESHOLD).mean(axis=0) * 100

    # Plant-averaged AUDPC (trapezoidal integration)
    audpc_t = np.zeros((n, len(TIME_POINTS)))
    for i in range(1, len(TIME_POINTS)):
        dt = TIME_POINTS[i] - TIME_POINTS[i - 1]
        audpc_t[:, i] = audpc_t[:, i-1] + 0.5 * (sev[:, i] + sev[:, i-1]) * dt
    mean_audpc = audpc_t.mean(axis=0)

    # Rates of change per 60-day period
    rate_inc   = np.diff(inc,        prepend=inc[0])
    rate_audpc = np.diff(mean_audpc, prepend=mean_audpc[0])
    rate_mort  = np.diff(mort,       prepend=mort[0])

    print(f"  [{label}]  D1200 →  "
          f"incidence = {inc[-1]:.1f}%  |  "
          f"AUDPC = {mean_audpc[-1]:.1f}  |  "
          f"mortality = {mort[-1]:.1f}%")

    return dict(label=label, months=months,
                inc=inc, audpc=mean_audpc, mort=mort,
                rate_inc=rate_inc, rate_audpc=rate_audpc, rate_mort=rate_mort)


# =============================================================================
# 3.  EPIDEMIOLOGICAL MODELS
# =============================================================================

def _monomolecular(t, K, r):   return K * (1 - np.exp(-r * t))
def _exponential(t, y0, r):    return y0 * np.exp(r * t)
def _logistic(t, K, r, t0):    return K / (1 + np.exp(-r * (t - t0)))
def _gompertz(t, K, r, t0):    return K * np.exp(-np.exp(-r * (t - t0)))
def _weibull(t, K, lam, k):
    tp = np.maximum(t, 1e-9); return K * (1 - np.exp(-(tp / lam) ** k))
def _log_logistic(t, K, a, b):
    tp = np.maximum(t, 1e-9); return K / (1 + (tp / a) ** (-b))

# {name: (function, lower_bounds, upper_bounds, initial_params)}
MODELS = {
    "Monomolecular": (_monomolecular, [0, 0],      [1.5, 2],      [0.80, 0.10]),
    "Exponential":   (_exponential,   [0, 0],      [0.5, 1],      [0.01, 0.05]),
    "Logistic":      (_logistic,      [0, 0, 0],   [1.5, 2, 50],  [0.80, 0.30, 20.0]),
    "Gompertz":      (_gompertz,      [0, 0, 0],   [1.5, 2, 50],  [0.80, 0.20, 15.0]),
    "Weibull":       (_weibull,       [0,.1,.1],   [1.5,100,10],  [0.80, 15.0, 1.5]),
    "Log-logistic":  (_log_logistic,  [0,.1,.1],   [1.5,100,20],  [0.80, 15.0, 2.0]),
}

MOD_COLORS = {
    "Monomolecular": "#555555",  "Exponential":  "#E67E22",
    "Logistic":      "#1B4F8A",  "Gompertz":     "#8E44AD",
    "Weibull":       "#C0392B",  "Log-logistic": "#1E8449",
}
MOD_LS = {
    "Monomolecular": "-",   "Exponential": "--",
    "Logistic":      "-",   "Gompertz":    "--",
    "Weibull":       "-.",  "Log-logistic":":",
}


def fit_models(t_months: np.ndarray, y_obs_raw: np.ndarray) -> list:
    """
    Fit all six models.  Data are normalised to [0, 1] before fitting
    and back-transformed afterward.
    Returns a list of result dicts (one per model).
    """
    max_v  = y_obs_raw.max()
    y_norm = np.clip(y_obs_raw / max_v, 0, 1)
    results = []
    for name, (func, lb, ub, p0) in MODELS.items():
        try:
            popt, _ = curve_fit(func, t_months, y_norm,
                                p0=p0, bounds=(lb, ub), maxfev=8000)
            pred   = func(t_months, *popt) * max_v
            ss_res = np.sum((y_obs_raw - pred) ** 2)
            ss_tot = np.sum((y_obs_raw - y_obs_raw.mean()) ** 2)
            r2     = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            rmse   = np.sqrt(ss_res / len(t_months))
            aic    = (len(t_months) * np.log(ss_res / len(t_months) + 1e-12)
                      + 2 * len(popt))
            results.append(dict(Model=name, R2=round(r2, 4), RMSE=round(rmse, 4),
                                AIC=round(aic, 2), params=popt, y_pred=pred))
        except Exception:
            results.append(dict(Model=name, R2=np.nan, RMSE=np.nan,
                                AIC=np.nan, params=None, y_pred=None))
    return results


def fit_all_lots(curves_list: list) -> tuple:
    """
    Run model fitting for Incidence (%) and AUDPC across all lots.
    Returns (fit_results dict, metrics_rows list).
    """
    fit_results  = {}
    metrics_rows = []
    for c in curves_list:
        label = c["label"]
        fit_results[label] = {}
        t_fit = np.array(c["months"][1:])          # exclude t = 0

        for var_key, y_full, var_label in [
            ("inc",   c["inc"],   "Incidence (%)"),
            ("audpc", c["audpc"], "AUDPC"),
        ]:
            y_obs = y_full[1:]
            res   = fit_models(t_fit, y_obs)
            fit_results[label][var_label] = res
            valid = [r for r in res if not np.isnan(r["AIC"])]
            best  = min(valid, key=lambda x: x["AIC"]) if valid else {}
            print(f"  [{label}]  {var_label:14s}  "
                  f"best = {best.get('Model','—'):14s}  "
                  f"R² = {best.get('R2',np.nan):.4f}  "
                  f"AIC = {best.get('AIC',np.nan):.1f}")
            for r in res:
                metrics_rows.append(dict(
                    Lot=label, Variable=var_label, Model=r["Model"],
                    **{"R²": r["R2"], "RMSE": r["RMSE"], "AIC": r["AIC"]}
                ))
    return fit_results, metrics_rows


# =============================================================================
# 4.  GLOBAL PLOT STYLE
# =============================================================================

plt.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["DejaVu Sans", "Helvetica", "Arial"],
    "font.size":         10,
    "axes.linewidth":    0.8,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "xtick.major.width": 0.8,  "ytick.major.width": 0.8,
    "xtick.major.size":  3.5,  "ytick.major.size":  3.5,
    "xtick.direction":   "out","ytick.direction":   "out",
    "pdf.fonttype":      42,   "ps.fonttype":        42,
})

LOT_CLR = {
    "Donmatias-L1": "#1B4F8A",
    "El Retiro-L2": "#C0392B",
    "La Ceja-L3":   "#1E8449",
}
LOT_MK      = {"Donmatias-L1":"o","El Retiro-L2":"s","La Ceja-L3":"^"}
BLACK       = "#000000"
LGRAY       = "#CCCCCC"
ALPHA_FILL  = 0.18


def _save(fig, basename: str) -> None:
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUTPUT_DIR, f"{basename}.{ext}"),
                    dpi=DPI if ext == "png" else None,
                    bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {basename}.png / .pdf")


# =============================================================================
# 5.  FIGURE 1 — CUMULATIVE CURVES + RATE OF CHANGE
# =============================================================================

def figure_cumulative(curves_list: list) -> None:
    """
    Layout: 3 rows (Incidence / AUDPC / Mortality) × 3 cols (lots).
    Left axis  → cumulative curve.
    Right axis → 60-day rate of change (dashed + shaded area).
    Lot names appear only as column headers (top row). All text black.
    No internal panel letters.
    """
    rows_cfg = [
        ("inc",   "rate_inc",   "Cumulative incidence (%)",   "Rate (% per 60 d)"),
        ("audpc", "rate_audpc", "Cumulative AUDPC",            "AUDPC rate (per 60 d)"),
        ("mort",  "rate_mort",  "Cumulative mortality (%)",    "Rate (% per 60 d)"),
    ]
    months_arr = np.array(curves_list[0]["months"])

    fig = plt.figure(figsize=(18, 13), facecolor="white")
    gs  = gridspec.GridSpec(3, 3, figure=fig,
                             hspace=0.28, wspace=0.36,
                             left=0.07, right=0.97,
                             top=0.93,  bottom=0.07)

    for row, (cum_key, rate_key, ylabel_cum, ylabel_rate) in enumerate(rows_cfg):
        for col, c in enumerate(curves_list):
            clr = LOT_CLR[c["label"]]
            mk  = LOT_MK[c["label"]]
            ax  = fig.add_subplot(gs[row, col])
            ax2 = ax.twinx()

            # Right axis — rate of change
            ax2.plot(months_arr, c[rate_key],
                     color=clr, lw=1.2, ls="--", alpha=0.80, zorder=2)
            ax2.fill_between(months_arr, 0, c[rate_key],
                             color=clr, alpha=ALPHA_FILL, zorder=1)
            ax2.set_ylabel(ylabel_rate, fontsize=10, color=BLACK,
                           labelpad=3, rotation=270, va="bottom")
            ax2.tick_params(axis="y", labelcolor=BLACK, labelsize=9)
            ax2.yaxis.set_major_locator(mticker.MaxNLocator(4))
            ax2.spines["right"].set_color(LGRAY)
            ax2.spines["right"].set_linewidth(0.7)
            ax2.spines["left"].set_visible(False)

            # Left axis — cumulative curve
            ax.plot(months_arr, c[cum_key], color=clr, lw=2.0, zorder=4)
            ax.scatter(months_arr[::2], c[cum_key][::2],
                       color=clr, marker=mk, s=28, zorder=5, linewidths=0)
            if col == 0:
                ax.set_ylabel(ylabel_cum, fontsize=10, color=BLACK, labelpad=3)
            ax.tick_params(axis="y", labelcolor=BLACK, labelsize=9)
            ax.yaxis.set_major_locator(mticker.MaxNLocator(5))
            ax.set_ylim(bottom=0)
            ax.set_axisbelow(True)
            ax.yaxis.grid(True, color=LGRAY, lw=0.4, ls=":")
            ax.set_facecolor("#FAFAFA")
            ax.set_xlabel("Time (months)", fontsize=10, labelpad=3, color=BLACK)
            ax.set_xlim(0, max(months_arr))
            ax.xaxis.set_major_locator(mticker.MultipleLocator(6))
            ax.tick_params(axis="x", labelsize=9, colors=BLACK)
            ax.spines["left"].set_color(LGRAY);   ax.spines["left"].set_linewidth(0.7)
            ax.spines["bottom"].set_color(LGRAY); ax.spines["bottom"].set_linewidth(0.7)

            # Column header — top row only
            if row == 0:
                ax.set_title(c["label"], fontsize=11,
                             fontweight="bold", color=BLACK, pad=6)

    fig.suptitle(
        "Cumulative epidemiological curves and rate of change — "
        "Incidence (%), AUDPC and Mortality (%)",
        fontsize=12, fontweight="bold", y=0.995, color=BLACK)
    _save(fig, "Fig1_Cumulative")


# =============================================================================
# 6.  FIGURE 2 — MODEL FITS  (A = Incidence · B = AUDPC)
# =============================================================================

def figure_model_fits(curves_list: list, fit_results: dict) -> None:
    """
    Layout: 2 rows × 3 cols.
    Internal R²-sorted legend per panel; y-axis extended 32 % to avoid overlap.
    Panel letters (A / B) on the outermost left column only.
    """
    VAR_ROWS = [
        ("Incidence (%)", "inc",   "Incidence (%)",    "A"),
        ("AUDPC",         "audpc", "Cumulative AUDPC", "B"),
    ]
    months_arr = np.array(curves_list[0]["months"])
    t_fit      = months_arr[1:]

    fig = plt.figure(figsize=(18, 12), facecolor="white")
    gs  = gridspec.GridSpec(2, 3, figure=fig,
                             hspace=0.13, wspace=0.22,
                             left=0.08, right=0.99,
                             top=0.94,  bottom=0.08)

    for row, (vl, ck, ylabel, panel_letter) in enumerate(VAR_ROWS):
        for col, c in enumerate(curves_list):
            ax    = fig.add_subplot(gs[row, col])
            y_obs = c[ck][1:]

            ax.set_ylim(0, y_obs.max() * 1.32)

            # Observed field data
            ax.scatter(t_fit, y_obs, color="black",
                       s=18, zorder=6, marker="o", linewidths=0)

            res = fit_results[c["label"]][vl]
            leg_handles = [
                Line2D([0],[0], marker="o", color="w",
                       markerfacecolor="black", markersize=7,
                       lw=0, label="Observed (field data)")
            ]
            for r in sorted(res,
                            key=lambda x: x["AIC"] if not np.isnan(x["AIC"]) else 9999):
                if r["y_pred"] is None: continue
                r2s = f"{r['R2']:.3f}" if not np.isnan(r["R2"]) else "n/a"
                ax.plot(t_fit, r["y_pred"],
                        color=MOD_COLORS[r["Model"]],
                        lw=1.8, ls=MOD_LS[r["Model"]],
                        zorder=3, alpha=0.90)
                leg_handles.append(
                    Line2D([0],[0], color=MOD_COLORS[r["Model"]],
                           lw=1.6, ls=MOD_LS[r["Model"]],
                           label=f"{r['Model']} (R²={r2s})"))

            leg = ax.legend(handles=leg_handles, fontsize=9.5,
                            loc="upper left", framealpha=0.94,
                            edgecolor=LGRAY, fancybox=False,
                            borderpad=0.7, handlelength=1.8,
                            handletextpad=0.45, labelspacing=0.28)
            leg.get_frame().set_linewidth(0.6)

            ax.set_facecolor("#FAFAFA"); ax.set_axisbelow(True)
            ax.yaxis.grid(True, color=LGRAY, lw=0.4, ls=":")
            ax.set_xlim(0, max(months_arr))
            ax.xaxis.set_major_locator(mticker.MultipleLocator(6))
            ax.yaxis.set_major_locator(mticker.MaxNLocator(5))
            ax.tick_params(labelsize=9, colors=BLACK, pad=3)
            ax.spines["left"].set_color(LGRAY);   ax.spines["left"].set_linewidth(0.7)
            ax.spines["bottom"].set_color(LGRAY); ax.spines["bottom"].set_linewidth(0.7)

            if row == 1:
                ax.set_xlabel("Time (months)", fontsize=10, labelpad=4, color=BLACK)
            else:
                ax.tick_params(axis="x", labelbottom=False)
            if col == 0:
                ax.set_ylabel(ylabel, fontsize=10, labelpad=4, color=BLACK)
            if row == 0:
                ax.set_title(c["label"], fontsize=11,
                             fontweight="bold", color=BLACK, pad=6)
            if col == 0:
                ax.text(-0.16, 1.06, panel_letter,
                        transform=ax.transAxes, fontsize=20,
                        fontweight="bold", color=BLACK, va="top")

    fig.suptitle(
        "Epidemiological model fits — "
        "Incidence (%, panel A) and Cumulative AUDPC (panel B)",
        fontsize=13, fontweight="bold", y=0.998, color=BLACK)
    _save(fig, "Fig2_ModelFits")


# =============================================================================
# 7.  WORD TABLE — MODEL SELECTION METRICS
# =============================================================================

def make_word_table(metrics_rows: list) -> None:
    """
    Export model fit metrics to a formatted landscape Word document.
    Colour-coded by lot; best model per lot × variable marked with ✓.
    """
    try:
        from docx import Document
        from docx.shared    import Pt, RGBColor, Inches
        from docx.enum.text import WD_ALIGN_PARAGRAPH
        from docx.oxml.ns   import qn
        from docx.oxml      import OxmlElement
    except ImportError:
        print("  python-docx not installed — skipping Word table."); return

    df = pd.DataFrame(metrics_rows)
    df["Best"] = False
    for (lot, var), grp in df.groupby(["Lot", "Variable"]):
        valid = grp[~grp["AIC"].isna()]
        if len(valid):
            df.loc[valid["AIC"].idxmin(), "Best"] = True
    df = df.sort_values(["Lot", "Variable", "AIC"]).reset_index(drop=True)

    LC = {
        "Donmatias-L1": {"row":"D6E4F0","best":"AED6F1","hdr":"1B4F8A"},
        "El Retiro-L2": {"row":"FADBD8","best":"F1948A","hdr":"C0392B"},
        "La Ceja-L3":   {"row":"D5F5E3","best":"ABEBC6","hdr":"1E8449"},
    }

    def rgb(h): return RGBColor(int(h[:2],16), int(h[2:4],16), int(h[4:],16))
    def bg(cell, hc):
        tc=cell._tc; p=tc.get_or_add_tcPr()
        s=OxmlElement("w:shd"); s.set(qn("w:val"),"clear")
        s.set(qn("w:color"),"auto"); s.set(qn("w:fill"),hc); p.append(s)
    def brd(cell, color="FFFFFF", sz=4):
        tc=cell._tc; p=tc.get_or_add_tcPr(); b=OxmlElement("w:tcBorders")
        for e in ["top","left","bottom","right"]:
            el=OxmlElement(f"w:{e}"); el.set(qn("w:val"),"single")
            el.set(qn("w:sz"),str(sz)); el.set(qn("w:color"),color); b.append(el)
        p.append(b)
    def cell_text(cell, text, bold=False, italic=False,
                  sz=9, align=WD_ALIGN_PARAGRAPH.CENTER, color="1C2833"):
        pp=cell.paragraphs[0]; pp.alignment=align
        pp.paragraph_format.space_before=Pt(1)
        pp.paragraph_format.space_after =Pt(1)
        r=pp.add_run(str(text)); r.bold=bold; r.italic=italic
        r.font.size=Pt(sz); r.font.color.rgb=rgb(color)

    doc = Document()
    sec = doc.sections[0]
    sec.page_width=Inches(11); sec.page_height=Inches(8.5)
    for attr in ["top_margin","bottom_margin","left_margin","right_margin"]:
        setattr(sec, attr, Inches(0.6))

    # Title
    p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
    r=p.add_run("Table. Epidemiological model fit comparison — "
                "Incidence (%) and Severity (AUDPC)")
    r.bold=True; r.font.size=Pt(12); r.font.color.rgb=rgb("1C2833")

    p2=doc.add_paragraph(); p2.alignment=WD_ALIGN_PARAGRAPH.CENTER
    r2=p2.add_run("PRR field survey — Donmatias-L1 · El Retiro-L2 · La Ceja-L3  |  "
                  "R², RMSE, AIC  |  ✓ = best model per lot × variable (lowest AIC)")
    r2.italic=True; r2.font.size=Pt(9.5); r2.font.color.rgb=rgb("666666")
    doc.add_paragraph()

    col_headers = ["Lot","Variable","Model","R²","RMSE","AIC","✓"]
    table = doc.add_table(rows=1, cols=len(col_headers))
    table.style = "Table Grid"
    for ci, lbl in enumerate(col_headers):
        c=table.rows[0].cells[ci]; bg(c,"1C2833"); brd(c)
        cell_text(c, lbl, bold=True, sz=9, color="FFFFFF")

    cur_lot=None; cur_var=None
    for _, row in df.iterrows():
        lc       = LC.get(row["Lot"], {"row":"F5F5F5","best":"DDDDDD","hdr":"555555"})
        is_best  = bool(row["Best"])

        # Lot separator
        if row["Lot"] != cur_lot:
            sep=table.add_row(); mc=sep.cells[0].merge(sep.cells[-1])
            bg(mc, lc["hdr"]); brd(mc, lc["hdr"], 6)
            cell_text(mc, row["Lot"], bold=True, sz=10,
                      align=WD_ALIGN_PARAGRAPH.LEFT, color="FFFFFF")
            cur_lot=row["Lot"]; cur_var=None

        # Variable sub-separator
        if row["Variable"] != cur_var:
            per=table.add_row(); mc2=per.cells[0].merge(per.cells[-1])
            bg(mc2,"F2F2F2"); brd(mc2,"AAAAAA",3)
            cell_text(mc2, row["Variable"], bold=True, italic=True, sz=9,
                      align=WD_ALIGN_PARAGRAPH.LEFT, color="444444")
            cur_var=row["Variable"]

        # Data row
        dr   = table.add_row()
        fill = lc["best"] if is_best else lc["row"]
        vals = [
            row["Lot"], row["Variable"], row["Model"],
            f"{row['R²']:.4f}"  if not pd.isna(row["R²"])   else "—",
            f"{row['RMSE']:.4f}" if not pd.isna(row["RMSE"]) else "—",
            f"{row['AIC']:.2f}"  if not pd.isna(row["AIC"])  else "—",
            "✓" if is_best else "",
        ]
        for ci, (val, cell) in enumerate(zip(vals, dr.cells)):
            bg(cell, fill); brd(cell)
            align = (WD_ALIGN_PARAGRAPH.LEFT if ci <= 1
                     else WD_ALIGN_PARAGRAPH.CENTER)
            cell_text(cell, val, bold=is_best, sz=9, align=align)

    note=doc.add_paragraph()
    rn=note.add_run(
        "Note: Models sorted by AIC within each lot × variable combination. "
        "R² = coefficient of determination; RMSE = root mean square error; "
        "AIC = Akaike Information Criterion. "
        "Field data from commercial avocado production lots in Antioquia, Colombia.")
    rn.italic=True; rn.font.size=Pt(8.5); rn.font.color.rgb=rgb("666666")

    out=os.path.join(OUTPUT_DIR,"Table_ModelFits.docx")
    doc.save(out); print(f"  Saved: Table_ModelFits.docx")


# =============================================================================
# 8.  MAIN PIPELINE
# =============================================================================

if __name__ == "__main__":

    # 1. Load all lots from the unified workbook
    dfs = load_all_lots()
    if not dfs:
        raise RuntimeError("No lots loaded. Check UNIFIED_XLSX and SHEETS.")

    # 2. Compute epidemiological curves
    print("\n── Epidemiological curves ──")
    curves_list = [compute_curves(df, label) for label, df in dfs.items()]

    # 3. Figure 1 — cumulative curves
    print("\n── Figure 1: Cumulative curves ──")
    figure_cumulative(curves_list)

    # 4. Model fitting
    print("\n── Model fitting ──")
    fit_results, metrics_rows = fit_all_lots(curves_list)
    pd.DataFrame(metrics_rows).to_csv(
        os.path.join(OUTPUT_DIR, "model_fit_metrics.csv"), index=False)
    print("  Saved: model_fit_metrics.csv")

    # 5. Figure 2 — model fits
    print("\n── Figure 2: Model fits ──")
    figure_model_fits(curves_list, fit_results)

    # 6. Word table
    print("\n── Word table ──")
    make_word_table(metrics_rows)

    print("\n✓  All temporal analyses complete.")